# 주제 ③ 프레스 유압펌프 진동·전류 시계열 — 데이터 품질 진단 (01_data_quality)

**목적**: 주제 선정을 위해 데이터 특성과 필요한 전처리(결측·중복·이상치·불균형·누수)를 파악한다. 심사기준 1번 대응. 모델링은 하지 않는다.

**데이터**: `data/3. 소성가공 예지보전 AI 데이터셋.zip` → `data/raw/` (CSV 2개, 가이드북 없음)

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
import data_quality as dq
FIG, RES = paths.nb_dirs("01_data_quality")
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 60)
def save(name): plt.tight_layout(); plt.savefig(FIG / name, dpi=110); plt.close(); print("saved", name)

## 0단계. 데이터 구성 파악

In [2]:
display(dq.file_inventory())
dfs = dq.load_all(); N, O = dfs["normal"], dfs["outlier"]
for k, d in dfs.items():
    print(f"{k:8s} shape={d.shape}  {d.ts.min()} ~ {d.ts.max()}  span={d.ts.max()-d.ts.min()}  label={d[dq.LABEL].unique()}")
print("columns:", list(pd.read_csv(dq.RAW_DIR / dq.FILES['normal'], nrows=1).columns))
display(N[dq.SENSORS + [dq.LABEL]].describe().T.round(3)); display(O[dq.SENSORS + [dq.LABEL]].describe().T.round(3))

,key,file,size_MB,encoding
0,normal,press_data_normal.csv,1.212,utf-8
1,outlier,outlier_data.csv,0.039,utf-8


normal   shape=(20000, 9)  2022-07-12 00:00:00.019000 ~ 2022-07-12 01:16:55.828000  span=0 days 01:16:55.809000  label=[0]
outlier  shape=(600, 9)  2022-07-17 10:51:07.943000 ~ 2022-07-17 10:53:53.540000  span=0 days 00:02:45.597000  label=[1]
columns: ['Unnamed: 0', 'TimeStamp', 'AI0_Vibration', 'AI1_Vibration', 'AI2_Current', 'Equipment_state']


,count,mean,std,min,25%,50%,75%,max
AI0_Vibration,20000.0,-0.000,0.071,-0.315,-0.044,-0.000,0.044,0.352
AI1_Vibration,20000.0,-0.000,0.119,-0.366,-0.069,-0.009,0.060,0.400
AI2_Current,20000.0,1.446,122.664,-271.568,-101.312,1.850,103.844,273.235
Equipment_state,20000.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000


,count,mean,std,min,25%,50%,75%,max
AI0_Vibration,600.0,0.008,0.448,-1.503,-0.116,0.016,0.093,1.803
AI1_Vibration,600.0,-0.008,0.241,-0.751,-0.136,0.002,0.171,0.509
AI2_Current,600.0,57.330,153.173,-399.351,-33.379,53.644,146.627,538.826
Equipment_state,600.0,1.000,0.000,1.000,1.000,1.000,1.000,1.000


**0단계 결론**
- CSV 2개(UTF-8). 컬럼: 원본 인덱스, `TimeStamp`(ms 단위), 센서 3개(`AI0_Vibration` 상부 진동, `AI1_Vibration` 하부 진동, `AI2_Current` 모터 전류 — 문제 설명 기준, 추정), `Equipment_state`(0=정상, 1=이상).
- `press_data_normal.csv` 20,000행 전부 0, `outlier_data.csv` 600행 전부 1 → **파일 = 라벨**. 정상은 2022-07-12 00:00~01:17(77분), 이상은 **2022-07-17 10:51~10:54(2분 46초)** 로 **날짜가 다르다**.
- 학습/테스트 분할 없음. 설비 ID 없음(단일 설비 1대, 추정). 가이드북 없음 → 진동 단위(g? mm/s?), 전류 단위(A? raw ADC?) 불명.

## 1단계. 데이터 품질 진단
### 1-1. 기본 구조·결측·중복

In [3]:
for k, d in dfs.items():
    print(f"== {k}: NaN={d[dq.SENSORS].isna().sum().sum()}  dup rows={d.duplicated(subset=['TimeStamp']+dq.SENSORS).sum()}  dup ts={d.ts.duplicated().sum()}  ts monotonic={d.ts.is_monotonic_increasing}  nunique={d[dq.SENSORS].nunique().to_dict()}")

== normal: NaN=0  dup rows=1  dup ts=1  ts monotonic=True  nunique={'AI0_Vibration': 19124, 'AI1_Vibration': 19460, 'AI2_Current': 19934}
== outlier: NaN=0  dup rows=0  dup ts=0  ts monotonic=True  nunique={'AI0_Vibration': 594, 'AI1_Vibration': 597, 'AI2_Current': 340}


### 1-2. 생산단위 식별
- **1행 = 1샘플(0.1초)**. 라벨은 샘플 단위로 붙어 있지만 값은 파일 단위로 동일 → 실질적으로 **구간(세그먼트) 단위 라벨**.
- 설비·라인·제품 정보 없음. 계층 구조는 "날짜 → 수집 burst(세그먼트) → 샘플" 뿐.

### 1-3. 시간 구조: 샘플링 간격, 세그먼트(burst), 공백

In [4]:
for k, d in dfs.items():
    dt = d.dt.dropna()
    print(f"== {k}: dt 최빈값={dt.round(3).mode()[0]}s | dt>0.2s 공백 {int((dt>0.2).sum())}개 | 최대 공백 {dt.max():.1f}s | 세그먼트 수 {d.seg.nunique()}")
segN, segO = dq.segment_table(N), dq.segment_table(O)
print("normal 세그먼트 길이 분포(샘플):"); display(segN.n.describe().round(1).to_frame().T)
print("outlier 세그먼트:"); display(segO[["n","start","end","dur_s","v0_sd","v1_sd","i_sd","v0_absmax"]])

== normal: dt 최빈값=0.1s | dt>0.2s 공백 598개 | 최대 공백 16.6s | 세그먼트 수 599
== outlier: dt 최빈값=0.1s | dt>0.2s 공백 20개 | 최대 공백 8.6s | 세그먼트 수 21
normal 세그먼트 길이 분포(샘플):


,count,mean,std,min,25%,50%,75%,max
n,599.0,33.4,16.4,1.0,20.0,37.0,50.0,50.0


outlier 세그먼트:


,n,start,end,dur_s,v0_sd,v1_sd,i_sd,v0_absmax
seg,,,,,,,,
0,4,2022-07-17 10:51:07.943,2022-07-17 10:51:08.243,0.3,0.424224,0.515501,60.145594,1.070320
1,50,2022-07-17 10:51:16.724,2022-07-17 10:51:21.624,4.9,0.394928,0.255612,89.645969,1.014148
2,47,2022-07-17 10:51:23.615,2022-07-17 10:51:28.215,4.6,0.516847,0.283511,102.708831,1.184883
3,31,2022-07-17 10:51:31.734,2022-07-17 10:51:34.734,3.0,0.029136,0.065331,67.670350,0.112750
4,18,2022-07-17 10:51:39.999,2022-07-17 10:51:41.699,1.7,0.708404,0.302478,98.267544,1.214663
5,3,2022-07-17 10:51:48.093,2022-07-17 10:51:48.293,0.2,0.579918,0.420179,22.992260,1.275405
6,50,2022-07-17 10:51:56.684,2022-07-17 10:52:01.584,4.9,0.562328,0.253219,131.031999,1.552577
7,40,2022-07-17 10:52:04.407,2022-07-17 10:52:08.307,3.9,0.338721,0.230732,134.595481,0.982635
8,25,2022-07-17 10:52:12.567,2022-07-17 10:52:14.967,2.4,0.328920,0.250760,93.064096,0.744857


In [5]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].plot(N.ts, N.AI0_Vibration, lw=.3); axes[0].set_title("normal(07-12): AI0_Vibration 전체 77분 — 세로 공백이 burst 사이 간격")
axes[1].plot(O.ts, O.AI0_Vibration, lw=.5, c="crimson"); axes[1].set_title("outlier(07-17): AI0_Vibration 전체 2분46초 (21 세그먼트)")
save("timeline_vibration.png")
fig, ax = plt.subplots(figsize=(12, 3)); ax.hist(segN.n, bins=50, color="steelblue"); ax.set_title("normal: 세그먼트 길이(샘플 수) 분포 — 최대 50샘플(5초) 단위 burst 수집"); save("segment_length_hist.png")

saved timeline_vibration.png
saved segment_length_hist.png


**시간 구조 진단 결과**
- 샘플링 **10 Hz(0.1 s)**. 연속 스트림이 아니라 **최대 50샘플(5초) burst 단위**로 수집되고 burst 사이에 2~17초 공백 → normal 599개, outlier 21개 세그먼트.
- 이상 데이터는 단 하루, 2분 46초, 21 burst. 정상은 다른 날 77분. → **시간·날짜 자체가 라벨과 1:1** (누수, 1-7).
- 이동 윈도우 피처를 만들 때 세그먼트 경계를 넘으면 안 된다(`dq.rolling_rms`는 세그먼트별로 계산).

### 1-4. 파형 특성: 전류는 AC 파형이며 10 Hz 샘플링으로 에일리어싱된 상태

In [6]:
fig, axes = plt.subplots(2, 1, figsize=(14, 5))
s = N.iloc[:100]; axes[0].plot(s.ts, s.AI2_Current, marker=".", lw=.8); axes[0].set_title("normal 첫 10초 AI2_Current — 겉보기 주기 ~1.5초의 정현파")
s = O[O.seg == 1]; axes[1].plot(s.ts, s.AI2_Current, marker=".", lw=.8, c="crimson"); axes[1].set_title("outlier 세그먼트1 (5초) AI2_Current")
save("current_waveform.png")
print("겉보기 주기(샘플): normal", round(dq.zero_crossing_period(N.AI2_Current.values), 2), "| outlier", round(dq.zero_crossing_period(O.AI2_Current.values), 2))
print("normal 세그먼트별 겉보기 주기 sd:", round(N.groupby('seg').AI2_Current.apply(lambda x: dq.zero_crossing_period(x.values)).replace(np.inf, np.nan).std(), 2))

saved current_waveform.png
겉보기 주기(샘플): normal 15.26 | outlier 9.6
normal 세그먼트별 겉보기 주기 sd: 3.02


- 전류가 ±270 사이를 오가는 **정현파**. 실제 모터 전류는 60 Hz AC인데 샘플링이 10 Hz라 **에일리어싱된 겉보기 파형**(주기 약 15샘플=1.5초)이다 (추정, 근거: 나이키스트 5 Hz < 60 Hz). 따라서 **주파수 분석(FFT)은 무의미**하고, RMS·피크·엔벨로프 같은 진폭 통계만 신뢰할 수 있다.
- 순간값(샘플 1개)의 전류는 위상에 따라 −270~+270 어디든 올 수 있어 **샘플 단위 분류는 부적절** → 윈도우(세그먼트) 단위 요약 피처가 필수.
- 진동 두 채널은 평균 0 근처의 잡음형 신호. normal에서 AI0·AI1 상관 0.37, 전류와는 무상관.

### 1-5. 이상치 및 정상/이상 분포 비교

In [7]:
print("normal 기준 |z|>3 비율:"); display(pd.concat({"normal": dq.outlier_table(N, N), "outlier": dq.outlier_table(O, N)}, axis=1))
rN, rO = dq.rolling_rms(N), dq.rolling_rms(O)
print("이동 RMS(1초) 분위수 — normal vs outlier:")
display(pd.concat({"normal": rN.quantile([.05, .5, .95, .99]).T, "outlier": rO.quantile([.05, .5, .95, .99]).T}, axis=1).round(3))

normal 기준 |z|>3 비율:


normal                          outlier                  
              share_abs_z_gt      min      max share_abs_z_gt      min      max
AI0_Vibration          0.005   -0.315    0.352         0.3417   -1.503    1.803
AI1_Vibration          0.001   -0.366    0.400         0.1350   -0.751    0.509
AI2_Current            0.000 -271.568  273.235         0.0367 -399.351  538.826

이동 RMS(1초) 분위수 — normal vs outlier:

normal                            outlier                           
                 0.05     0.50     0.95     0.99    0.05     0.50     0.95     0.99
AI0_Vibration   0.030    0.064    0.109    0.130   0.034    0.179    0.822    1.028
AI1_Vibration   0.034    0.087    0.203    0.218   0.056    0.189    0.368    0.395
AI2_Current    77.111  100.915  182.909  192.197  45.494  103.203  305.998  402.147

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, dq.SENSORS):
    sns.kdeplot(rN[c].dropna(), ax=ax, label="normal", fill=True); sns.kdeplot(rO[c].dropna(), ax=ax, label="outlier", fill=True, color="crimson")
    ax.set_title(f"이동 RMS(1s) 분포: {c}"); ax.legend()
save("rms_distribution.png")

saved rms_distribution.png


In [9]:
# normal 내부의 상태 변화(10분위 구간) / outlier 세그먼트 중 '조용한' 구간
N["chunk"] = pd.qcut(np.arange(len(N)), 10, labels=False)
display(N.groupby("chunk").agg(v0_sd=("AI0_Vibration","std"), v1_sd=("AI1_Vibration","std"), i_sd=("AI2_Current","std"), i_absmax=("AI2_Current", lambda x: x.abs().max())).round(3).T)
quiet = segO[segO.v0_sd < N.AI0_Vibration.std()]; print("outlier 세그먼트 중 진동 sd가 normal 전체 sd보다 작은 것:", quiet.index.tolist(), "(샘플", int(quiet.n.sum()), "개)")

chunk,0,1,2,3,4,5,6,7,8,9
v0_sd,0.069,0.074,0.074,0.076,0.074,0.077,0.076,0.074,0.039,0.064
v1_sd,0.110,0.124,0.118,0.136,0.128,0.133,0.144,0.126,0.044,0.094
i_sd,121.754,131.141,122.319,133.014,126.909,129.115,134.629,123.090,86.050,111.276
i_absmax,261.620,271.720,273.235,262.430,272.439,273.128,272.702,267.391,118.209,265.038


outlier 세그먼트 중 진동 sd가 normal 전체 sd보다 작은 것: [3, 19, 20] (샘플 56 개)


**이상치·분포 진단 결과**
- 이상 데이터의 진동은 normal 대비 **AI0 sd 6배(0.07→0.45), AI1 2배**, 이동 RMS 중앙값 3배. 전류 |값|>300은 normal 0%, outlier 8%. → 진폭 통계만으로도 구간 대부분은 분리된다.
- 그러나 **outlier 세그먼트 3, 19, 20(총 72샘플, 12%)은 진동 sd가 normal보다 작다** → 라벨이 "이상 발생 시간대"에 통째로 붙은 것이고, 샘플/세그먼트 단위로는 정상 구간이 섞여 있다. 이 구간이 모델의 **FN(미탐지)으로 집계될 것이며, 실제로는 라벨 노이즈**다.
- normal 8번째 10분위 구간(약 60~68분)은 전류 최대값 118, 진동 sd 절반 → 저부하/대기 상태로 추정. 정상에도 운전 상태가 여러 개 있으며, 이 구간을 임계값으로 이상 판정하면 **FP(오경보)** 후보.
- 물리 단위를 몰라 절대 임계값(예: ISO 10816 진동 기준)은 적용 불가.

### 1-6. 불균형

In [10]:
tot = len(N) + len(O); print(f"샘플: normal {len(N)} / outlier {len(O)} → 이상 비율 {len(O)/tot:.2%}")
print(f"세그먼트: normal {N.seg.nunique()} / outlier {O.seg.nunique()} → 이상 비율 {O.seg.nunique()/(N.seg.nunique()+O.seg.nunique()):.2%}")
print("이상 이벤트(날짜) 수: 1")
fig, ax = plt.subplots(figsize=(6, 3.2)); ax.bar(["normal", "outlier"], [len(N), len(O)], color=["steelblue", "crimson"]); ax.set_yscale("log"); ax.set_title("샘플 수 (log)"); save("label_imbalance.png")

샘플: normal 20000 / outlier 600 → 이상 비율 2.91%
세그먼트: normal 599 / outlier 21 → 이상 비율 3.39%
이상 이벤트(날짜) 수: 1
saved label_imbalance.png


- 샘플 기준 2.9%, 세그먼트 기준 3.4%, **이상 이벤트 기준 1건**. 통계적으로는 "정상 1일 + 이상 1회 촬영"이라 지도학습으로 일반화 성능을 주장하기 어렵다.
- 정상 데이터만으로 학습하는 **이상탐지(One-class·재구성 오차) 프레임 + 이상 데이터는 임계값 검증용**이 이 데이터의 구조에 맞는다.

### 1-7. 누수 위험

- **날짜/시간 = 라벨**: TimeStamp 또는 원본 인덱스를 피처로 넣으면 100% 분류. 절대 시간 피처 금지.
- **세그먼트 내부 자기상관**: 같은 burst의 샘플을 train/test에 나누면 인접 샘플이 답을 알려준다. 분할은 반드시 **세그먼트(또는 시간 블록) 단위**.
- **normal 파일이 하루치**: 요일·계절·다른 운전 조건이 없어 "정상 분포"가 너무 좁다. 다른 날의 정상을 이상으로 잡을 위험(오경보)이 데이터로 검증되지 않는다.
- 파형 위상(에일리어싱된 전류 순간값)은 우연히 파일별로 다를 수 있음 → 순간값 대신 RMS·피크 사용.

## 2단계. 진단 결과 → 필요 처리 정리

| 항목 | 필요도 | 근거 | 권장 방법 |
|---|---|---|---|
| 결측치 처리 | **하** | NaN 0, 중복 0 | 없음. 단, burst 사이 공백을 "결측 구간"으로 인식하고 윈도우가 넘지 않게 처리 |
| 중복 처리 | 하 | 완전 중복 없음(TimeStamp 중복 1건) | 없음 |
| 이상치 처리 | **중** | 이상 파일 안에 '조용한' 세그먼트(12%), 정상 안에 저부하 구간 | 라벨 정제: 이상 세그먼트별 진동 RMS로 **soft label** 또는 '조용한' 세그먼트 제외·별도 보고. 정상 저부하 구간은 운전 상태 플래그 |
| 불균형 처리 | **상** | 2.9%, 이상 이벤트 1건 | 오버샘플링은 의미 없음(같은 이벤트 복제). **정상만으로 학습하는 이상탐지**를 주 모델로, 지도 분류는 비교용 베이스라인 |
| 도메인 지식 결합 | **상** | 물리 단위 불명, 전류 에일리어싱 | 윈도우(1~5초) 피처: 진동 RMS·피크·첨도·왜도·crest factor, 전류 RMS·피크, 상·하부 진동 비(AI0/AI1), 세그먼트 내 추세. FFT 금지(10 Hz 샘플링). 조기탐지는 "세그먼트 시작 후 n샘플" 기준으로 탐지 지연을 측정 |
| 검증 전략 | **상** | 날짜=라벨, burst 자기상관 | **세그먼트 단위 GroupKFold** + normal은 시간 순 블록(앞 70%/뒤 30%)으로 오경보율 검증. 이상 21세그먼트는 leave-segment-out. 지표: 세그먼트 단위 recall(탐지율), 정상 세그먼트 FP율(오경보), 탐지 지연(샘플 수) |

### 주제 ③ 관점 총평
- **강점**: 데이터가 작고 깨끗해 파이프라인이 빠르다. 문제 자체가 "오경보 vs 미탐지" 트레이드오프라서 심사 3번(FN/FP 조건)·4번(사전경보)·5번(불확실성·임계값 보정) 서술 구조가 자연스럽다. 정상 내부 저부하 구간과 이상 내부 조용한 구간이 각각 FP·FN 분석의 구체적 재료가 된다.
- **리스크**: (1) 이상 이벤트가 **1건·1일·2분 46초** → 어떤 F1도 "이 사건 1개를 맞췄다"는 뜻이며, 일반화 주장이 불가. (2) 정상도 하루치라 오경보율 검증 근거가 약함. (3) 10 Hz 샘플링으로 진동·전류의 주파수 정보가 소실되어 예지보전 도메인 기법(스펙트럼, 베어링 결함 주파수)을 쓸 수 없음. (4) 물리 단위 불명 → 현장 임계값을 절대 단위로 못 씀.
- **난이도 한 줄 평**: 구현은 가장 쉽지만 **데이터 양이 너무 적어 "모델 성능"이 아니라 "탐지 지연·오경보 트레이드오프를 어떻게 설계·검증하는가"로 승부해야 하는 주제**. 심사 2번(40점)의 "2개 이상 모델 비교"는 가능하나 비교 결과의 통계적 의미가 약하다.